In [ ]:
from strats.trend_following_strategy_v2 import TrendFollowingStrategy
from models.backtesting_models import AccountState
from clients.hyperliquid_client import HyperliquidClient
from backtesting_engine import BacktestEngine
from risk_manager import RiskManager
from utils.misc import resample_ohlcv
from dataclasses import asdict
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import copy

import pandas as pd
from models.backtesting_models import TradeSignal

client = HyperliquidClient()

In [ ]:
# Load pre-fetched 1m data from parquet
market_bars_df = pd.read_parquet("")
# Convert to same dict structure as fetch_bars for compatibility with strategy
market_bars = {}
for symbol in market_bars_df['symbol'].unique():
    df_1m = market_bars_df[market_bars_df['symbol'] == symbol].reset_index(drop=True)
    df_1m["timestamp"] = pd.to_datetime(df_1m["timestamp"], utc=True)
    market_bars[symbol] = {"1m": df_1m, "15m": resample_ohlcv(df_1m, "15min"), "1h": resample_ohlcv(df_1m, "60min")}

In [ ]:
strategy = TrendFollowingStrategy()

account = AccountState(balance=10_000.0)
engine = BacktestEngine(allow_same_bar_exit=False)

risk_manager = RiskManager(
    leverage=5,
    risk_per_trade=0.02,
    maintenance_margin_rate=0.005,
    min_notional=100.0,
    max_absolute_margin=None,  # e.g. 200.0 for fixed $200 cash per trade
    tp_risk_reward_multiplier=1.5,
    
)

bars_1m_by_symbol = {sym: bars["1m"].set_index('timestamp') for sym, bars in market_bars.items() if "1m" in bars}
all_timestamps = market_bars_df['timestamp'].drop_duplicates().sort_values()

In [ ]:
signals = strategy.generate_trade_signals(market_bars)
print(f"Generated signals: {len(signals):,}")

In [ ]:
signals_by_timestamp: dict[pd.Timestamp, list[TradeSignal]] = defaultdict(list)
for sig in signals:
    ts = pd.to_datetime(sig.timestamp, utc=True)
    signals_by_timestamp[ts].append(sig)

In [ ]:
equity_history = []
all_rejected_signals = []
pending_execution_signals = {symbol: None for symbol in market_bars.keys()}


# Main simulation loop
for ts in all_timestamps:
    new_signals = signals_by_timestamp.get(ts, [])
    
    # 1. Get current market snapshot
    current_bars = {symbol: df.loc[ts] for symbol, df in bars_1m_by_symbol.items()}
    current_prices = {symbol: bar["close"] for symbol, bar in current_bars.items()}
    
    # 2. See if any existing trades closed on this bar
    account = engine.update_account(account, current_bars)   
    
    # 3. EXECUTE PENDING: Try to fill pending execution signals from previous bars first 
    # Sort pending signals by conviction (highest first) to prioritize stronger signals in case of margin constraints
    pending_execution_signals_sorted = dict(sorted(pending_execution_signals.items(), key=lambda item: item[1].conviction if item[1] else 0, reverse=True))
    still_pending = {symbol: None for symbol in market_bars.keys()}
    for symbol, exec_sig in pending_execution_signals_sorted.items():
        if exec_sig is None:
            # No pending signal for this symbol, skip
            continue
        bar = current_bars.get(symbol)
        if bar is not None:
            new_pos = engine.try_fill_execution(account, exec_sig, bar)
            if new_pos:
                account.open_positions.append(new_pos)
            else:
                if pd.to_datetime(exec_sig.timestamp, utc=True) + pd.Timedelta(minutes=15) > ts:
                    # Keep signal active for multiple bars (e.g. 15m)
                    still_pending[exec_sig.symbol] = exec_sig
                    
    pending_execution_signals = copy.deepcopy(still_pending)
    
    # 4. CALCULATE: Get fresh risk numbers based on current Equity
    current_equity = account.get_equity(current_prices)
    current_free_margin = account.get_free_margin(current_equity)
    
    # 5. RISK MANAGER: Size new signals based on the account's health
    if new_signals:
        execution_batch, rejected = risk_manager.build_execution_batch(
            signals=new_signals,
            equity=current_equity,
            free_margin=current_free_margin
        )
        # Overwrite old pending signals with new batch 
        for exec_sig in execution_batch:
            pending_signal_temp = pending_execution_signals.get(exec_sig.symbol)
            
            if pending_signal_temp is not None:
                if exec_sig.conviction > pending_signal_temp.conviction:
                    
                    pending_execution_signals[exec_sig.symbol] = copy.deepcopy(exec_sig)
            else:
                pending_execution_signals[exec_sig.symbol] = copy.deepcopy(exec_sig)
        # Track rejections across the whole backtest
        all_rejected_signals.extend(rejected)
    
                
    # 6. SNAPSHOT: Record equity and rejected signals
    current_equity = account.get_equity(current_prices)
    equity_history.append({
        "timestamp": ts,
        "equity": current_equity,
        "balance": account.balance,
        "open_count": len(account.open_positions)
    })
    

# Performance Reporting

In [ ]:
final_balance = account.balance
total_trades = account.closed_trades
equity_df = pd.DataFrame(equity_history) 
trades_df = pd.DataFrame([asdict(t) for t in total_trades])


wins = [t for t in total_trades if t.pnl > 0]
win_rate = (len(wins) / len(total_trades) * 100) if total_trades else 0
profit_factor = trades_df[trades_df['pnl'] > 0]['pnl'].sum() / abs(trades_df[trades_df['pnl'] < 0]['pnl'].sum())
trade_profit = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
trade_losses = trades_df[trades_df['pnl'] < 0]['pnl'].sum()

print(f"Final balance:    ${final_balance:,.2f}")
print(f"Total Trades:     {len(total_trades):,}")
print(f"Win rate:         {win_rate:.1f}%")
print(f"Open positions:   {len(account.open_positions):,}")
print(f"Profits/Losses:   ${trade_profit:,.2f} / ${trade_losses:,.2f}")
print(f"Profit Factor:   {profit_factor:.2f}")

In [ ]:
equity_df.set_index('timestamp')['equity'].plot(title="Equity Curve")

In [ ]:
bucket_labels = [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)]

conviction = (
    trades_df["conviction"]
    if "conviction" in trades_df.columns
    else trades_df["metadata"].map(lambda m: m.get("conviction") if isinstance(m, dict) else None)
)

conviction_df = trades_df.assign(
    conviction=pd.to_numeric(conviction, errors="coerce").clip(0, 1),
    is_win=trades_df["pnl"] > 0,
)
conviction_df["conviction_bucket"] = pd.cut(
    conviction_df["conviction"],
    bins=[i / 10 for i in range(11)],
    labels=bucket_labels,
    include_lowest=True,
    right=True,
)

bucket_stats = (
    conviction_df.groupby(["symbol", "conviction_bucket"], observed=False)
    .agg(
        trades=("pnl", "size"),
        avg_pnl=("pnl", "mean"),
        win_rate=("is_win", "mean"),
    )
    .reindex(bucket_labels, level="conviction_bucket")
)

bucket_stats["win_rate"] = (bucket_stats["win_rate"])
bucket_stats["avg_pnl"] = bucket_stats["avg_pnl"]

trades_map = bucket_stats["trades"].unstack("conviction_bucket").reindex(columns=bucket_labels)
avg_pnl_map = bucket_stats["avg_pnl"].unstack("conviction_bucket").reindex(columns=bucket_labels)
win_rate_map = bucket_stats["win_rate"].unstack("conviction_bucket").reindex(columns=bucket_labels)

plt.figure(figsize=(16, max(6, len(trades_map) * 0.22)))
sns.heatmap(
    trades_map,
    cmap="Blues",
    annot=True,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Trades"},
)
plt.title("Trade Count by Symbol and Conviction Bucket")
plt.xlabel("Conviction Bucket")
plt.ylabel("Symbol")
plt.show()

plt.figure(figsize=(18, max(6, len(avg_pnl_map) * 0.22)))
sns.heatmap(
    avg_pnl_map,
    cmap="RdYlGn",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Average PnL ($)"},
)
plt.title("Average PnL by Symbol and Conviction Bucket")
plt.xlabel("Conviction Bucket")
plt.ylabel("Symbol")
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, max(6, len(win_rate_map) * 0.22)))
sns.heatmap(
    win_rate_map,
    cmap="YlGn",
    vmin=0,
    vmax=100,
    annot=True,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Win Rate (%)"},
)
plt.title("Win Rate by Symbol and Conviction Bucket")
plt.xlabel("Conviction Bucket")
plt.ylabel("Symbol")
plt.show()

In [ ]:
# bucket_stats.to_clipboard()

In [ ]:
# Convert closed trades to a DataFrame
trades_df = pd.DataFrame([asdict(t) for t in account.closed_trades])


# Calculate 'R' (Realized Multiple)
# R = Realized PnL / Initial Risk Budget
trades_df['R_multiple'] = trades_df['pnl'] / trades_df['risk_budget']
    
# Clean up any potential infinite values
trades_df = trades_df.replace([float('inf'), float('-inf')], 0).dropna(subset=['R_multiple'])
    
def plot_trade_distribution(df):
    plt.figure(figsize=(10, 6))
    
    # Create the histogram
    sns.histplot(df['R_multiple'], bins=50, kde=True, color='skyblue')
    
    # Add vertical lines for context
    plt.axvline(0, color='red', linestyle='--', label='Break Even')
    plt.axvline(df['R_multiple'].mean(), color='green', linestyle='-', 
                label=f'Avg R: {df["R_multiple"].mean():.2f}')
    
    plt.title('Trade Distribution (R-Multiple)')
    plt.xlabel('Result in Units of Risk (R)')
    plt.ylabel('Frequency of Trades')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_trade_distribution(trades_df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Ensure exit_time is a datetime object and sorted
trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
trades_df = trades_df.sort_values('exit_time')

# 2. Re-calculate metrics based on chronological order
trades_df['cum_r'] = trades_df['R_multiple'].cumsum()
trades_df['peak'] = trades_df['cum_r'].cummax()
trades_df['drawdown'] = trades_df['cum_r'] - trades_df['peak']

# 3. Visualization with Time on the X-axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Equity Curve
ax1.plot(trades_df['exit_time'], trades_df['cum_r'], color='#2ca02c', label='Cumulative R', lw=2)
ax1.fill_between(trades_df['exit_time'], trades_df['cum_r'], alpha=0.1, color='#2ca02c')
ax1.set_title('Equity Curve (R-Multiple) Over Time', fontsize=14)
ax1.set_ylabel('R-Units')
ax1.grid(True, alpha=0.2)

# Drawdown Plot
ax2.fill_between(trades_df['exit_time'], trades_df['drawdown'], 0, color='#d62728', alpha=0.3)
ax2.plot(trades_df['exit_time'], trades_df['drawdown'], color='#d62728', lw=1)
ax2.set_title('Drawdown Over Time', fontsize=14)
ax2.set_ylabel('R-Units')
ax2.set_xlabel('Exit Date')
ax2.grid(True, alpha=0.2)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

frame = trades_df.copy()
frame["entry_time"] = pd.to_datetime(frame["entry_time"], utc=True, errors="coerce")
frame = frame.dropna(subset=["entry_time", "pnl"])
frame["entry_weekday"] = pd.Categorical(
    frame["entry_time"].dt.day_name(), categories=weekday_order, ordered=True
)
frame["entry_hour"] = frame["entry_time"].dt.hour
frame["is_win"] = frame["pnl"] > 0

stats = frame.groupby(["entry_weekday", "entry_hour"], observed=False).agg(
    avg_pnl=("pnl", "mean"),
    trade_count=("pnl", "size"),
    hit_rate=("is_win", "mean"),
)

avg_pnl_map = stats["avg_pnl"].unstack("entry_hour").reindex(weekday_order)
count_map = stats["trade_count"].unstack("entry_hour").reindex(weekday_order)
hit_rate_map = (stats["hit_rate"] * 100).unstack("entry_hour").reindex(weekday_order)

plt.figure(figsize=(14, 5))
sns.heatmap(
    avg_pnl_map,
    cmap="RdYlGn",
    center=0,
    annot=True,
    fmt=".0f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Avg PnL ($)"},
)
plt.title("Average PnL")
plt.xlabel("UTC hour")
plt.ylabel("")
plt.show()

plt.figure(figsize=(14, 5))
sns.heatmap(
    count_map,
    cmap="Blues",
    annot=True,
    fmt=".0f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Trades"},
)
plt.title("Number of Trades")
plt.xlabel("UTC hour")
plt.ylabel("")
plt.show()

plt.figure(figsize=(14, 5))
sns.heatmap(
    hit_rate_map,
    cmap="YlGn",
    vmin=0,
    vmax=100,
    annot=True,
    fmt=".1f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Positive Hit Rate (%)"},
)
plt.title("Positive Trade Hit Rate")
plt.xlabel("UTC hour")
plt.ylabel("")
plt.show()